# Guardrail Evaluation & Result Analysis

This notebook lets you **run guardrail evaluation** on labeled data, **inspect per-sample predictions**, and **analyze results** (metrics, confusion matrix, false positives/negatives).

- Load guardrails via `get_guardrails()` from the canonical submission module `src/submission/submission.py`.
- Run on a CSV with `content` + `label` (1 = harmful, 0 = safe).
- View precision, recall, F1, latency and a predictions table.
- Analyze errors: false positives (safe content flagged) and false negatives (harmful content missed).

**Command-line workflow:** Use `python -m src.guardrails.get_predictions` to write prediction CSVs, then `python -m src.guardrails.get_guardrail_metrics` to compute metrics from those files. In this notebook we run evaluation in one go and also show how to use `compute_metrics_from_predictions()` when you have predictions (e.g. from file).

## 1. Setup: path and imports

In [ ]:
import csv
import sys
from pathlib import Path

import pandas as pd

# Resolve project root robustly regardless of where the notebook is launched.
cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
elif (cwd / "src").exists() and (cwd / "scripts").exists():
    project_root = cwd  # already in project/
elif (cwd / "project").exists():
    project_root = cwd / "project"  # repo root
else:
    raise RuntimeError(f"Could not infer project root from cwd={cwd}")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.guardrails import (
    get_predictions,
    compute_metrics_from_predictions,
    load_guardrails_from_module,
    load_evaluation_data,
)
from src.guardrails.base import EvaluationType

print("Project root:", project_root)
print("Imports OK.")

## 2. Load guardrails and evaluation data

In [ ]:
# --- Config: canonical submission module path + evaluation dataset ---
SUBMISSION_MODULE = project_root / "src" / "submission" / "submission.py"
DATA_CSV = project_root.parent / "datasets" / "sample_guardrail_data.csv"

if not SUBMISSION_MODULE.exists():
    raise FileNotFoundError(f"Submission module not found: {SUBMISSION_MODULE}")
if not DATA_CSV.exists():
    raise FileNotFoundError(f"Data CSV not found: {DATA_CSV}")

input_guardrail, _ = load_guardrails_from_module(SUBMISSION_MODULE)
evaluation_data = load_evaluation_data(DATA_CSV)

print(f"Loaded {len(evaluation_data)} samples from {DATA_CSV.name}")
print(f"Input guardrail: {'yes' if input_guardrail else 'none'}")

## 3. Run evaluation (get predictions + metrics)

We run the guardrail on the data and get both per-sample predictions and aggregate metrics. This is equivalent to what `get_predictions` produces (predictions) plus what `get_guardrail_metrics` computes from them (metrics).

In [ ]:
# Evaluate input guardrail
guardrail = input_guardrail
if guardrail is None:
    raise ValueError("No input guardrail; check submission returns guardrails.")

# Print guardrail information
print(f"Guardrail: {guardrail.__class__.__name__}")
print(f"Guardrail type: {type(guardrail)}")

# Get predictions
predictions = get_predictions(
    guardrail,
    evaluation_data,
    evaluation_type=EvaluationType.USER_INPUT,
    include_latency=True,
    content_key="content",
    label_key="label",
)
metrics_result = compute_metrics_from_predictions(predictions)

print("Metrics (positive class = harmful)")
print("-" * 50)
print(f"  Precision: {metrics_result.precision:.4f}")
print(f"  Recall:    {metrics_result.recall:.4f}")
print(f"  F1:        {metrics_result.f1:.4f}")
print(f"  Support:   harmful={metrics_result.support_harmful}, safe={metrics_result.support_safe}")
if metrics_result.latency_ms_mean is not None:
    print(f"  Latency:   mean={metrics_result.latency_ms_mean:.2f} ms, total={metrics_result.latency_ms_total:.2f} ms")
print(f"  Guardrails: {metrics_result.guardrail_names}")

## 4. Predictions table

In [ ]:
# Build DataFrame (convert bools to 0/1 for readability)
df = pd.DataFrame(predictions)
for col in df.select_dtypes(include=["bool"]).columns:
    df[col] = df[col].astype(int)

df.index.name = "sample_id"
display(df)

## 5. Result analysis

In [ ]:
# Confusion matrix (ground truth vs combined prediction)
y_true = df["label_harmful"].astype(bool)
y_pred = df["combined_pred"].astype(bool)

tp = ((y_true) & (y_pred)).sum()
fp = ((~y_true) & (y_pred)).sum()
tn = ((~y_true) & (~y_pred)).sum()
fn = ((y_true) & (~y_pred)).sum()

print("Confusion matrix (rows=true, cols=pred)")
print("                 pred_safe  pred_harmful")
print(f"  true_safe      {tn:>6}      {fp:>6}")
print(f"  true_harmful   {fn:>6}      {tp:>6}")
print()
print(f"  TP={tp}  FP={fp}  TN={tn}  FN={fn}")

In [ ]:
# False positives: safe content incorrectly flagged as harmful
fp_mask = (~y_true) & (y_pred)
fp_df = df.loc[fp_mask, ["content", "label", "combined_pred"]].copy()
fp_df["error"] = "false_positive"
print(f"False positives ({len(fp_df)}): safe content flagged as harmful")
print("-" * 60)
if len(fp_df) > 0:
    display(fp_df)
else:
    print("None.")

In [ ]:
# False negatives: harmful content incorrectly allowed
fn_mask = (y_true) & (~y_pred)
fn_df = df.loc[fn_mask, ["content", "label", "combined_pred"]].copy()
fn_df["error"] = "false_negative"
print(f"False negatives ({len(fn_df)}): harmful content missed")
print("-" * 60)
if len(fn_df) > 0:
    display(fn_df)
else:
    print("None.")

In [ ]:
# Optional: latency distribution (if column exists)
df["latency_ms"].describe()